# Inicialização

In [1]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor
import torch.optim as optim
from torchmetrics.functional.classification import multiclass_f1_score
from sklearn.metrics import f1_score, confusion_matrix

import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import copy
from copy import deepcopy
from tqdm import tqdm
import time
import os
from sklearn.cluster import KMeans
from scipy.spatial import distance
from scipy.signal import find_peaks, sosfiltfilt, butter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [2]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [3]:
actisP = ['lying', 'sitting', 'standing', 'walking', 'running', 'cycling',
          'Nordic', 'watching TV', 'computer', 'car driving', 'ascending stairs', 'descending stairs',
          'vacuum', 'ironing', 'folding', 'house cleaning', 'soccer', 'rope jumping']
activity_id = [1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 16, 17, 18, 19, 20, 24]
users = ['10'+ x for x in np.arange(1,10).astype(str)]
posis = ['hand', 'chest', 'ankle']
path = 'C:\\Meu Drive\\Doutorado Unicamp\\Projeto\\Dataset\\PAMAP2_Dataset\\'
data = np.load(path+'Xydata.npz')
Xdata = data['Xdata']/20 # /160
ydata = data['ydata']
valid = list(np.unique(ydata[:,1]))
for i in range(ydata.shape[0]):
    ydata[i,1] = valid.index(ydata[i,1])

In [4]:
def spectral_perturb(x, gain_db=6., n_knots=9, generator=None, preserve_dc=True, eps=1e-12):
    """x: torch.Tensor (B,3,J), retorna mesma forma, dtype e energia por janela.
    Resposta em dB interpolada linearmente em frequencia, compartilhada nos 3 eixos.
    Os limites +/-gain_db valem ANTES da renormalizacao de energia.
    """
    if gain_db < 0: raise ValueError('gain_db deve ser >= 0')
    if x.ndim != 3: raise ValueError('Esperado (B,3,J)')
    if not x.is_floating_point(): raise TypeError('Entrada deve ser float')
    B,C,J=x.shape
    if J < 4 or n_knots < 2: raise ValueError('J>=4 e n_knots>=2')
    if gain_db == 0: return x.clone()
    # Gerador CPU permite reproducibilidade independentemente do dispositivo.
    knots = (2*torch.rand((B,1,n_knots),generator=generator)-1)*gain_db
    if preserve_dc: knots[:,:,0]=0.
    # interpolacao de magnitudes em dB ao longo dos bins positivos da rFFT
    db = torch.nn.functional.interpolate(knots, size=J//2+1, mode='linear',align_corners=True)
    gain = torch.pow(torch.tensor(10.,dtype=x.dtype,device=x.device), db.to(x.device,dtype=x.dtype)/20)
    spectrum = torch.fft.rfft(x,dim=-1)
    y = torch.fft.irfft(spectrum*gain,n=J,dim=-1)
    e0 = x.square().sum(dim=(1,2),keepdim=True)
    e1 = y.square().sum(dim=(1,2),keepdim=True)
    scale = torch.sqrt(e0/e1.clamp_min(eps))
    return y*scale

In [5]:
class ChangEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv1d(in_channels=3, out_channels=16, kernel_size=3)
        self.inst1 = nn.InstanceNorm1d(16, affine=True)
        self.drop1 = nn.Dropout(p=0.2)

        self.conv2 = nn.Conv1d(in_channels=16, out_channels=16, kernel_size=3)
        self.inst2 = nn.InstanceNorm1d(16, affine=True)
        self.drop2 = nn.Dropout(p=0.2)

        self.conv3 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=5, stride=4)
        self.inst3 = nn.InstanceNorm1d(32, affine=True)
        self.drop3 = nn.Dropout(p=0.2)

        self.conv4 = nn.Conv1d(in_channels=32, out_channels=32, kernel_size=3, stride=1)
        self.inst4 = nn.InstanceNorm1d(32, affine=True)
        self.drop4 = nn.Dropout(p=0.2)

        self.conv5 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, stride=4)
        self.inst5 = nn.InstanceNorm1d(64, affine=True)
        self.drop5 = nn.Dropout(p=0.2)

        self.conv6 = nn.Conv1d(in_channels=64, out_channels=100, kernel_size=5, stride=1)

        self.relu = nn.LeakyReLU(0.3)
        self.glap = nn.AvgPool1d(kernel_size=4)

    def forward(self, x):
        # (N,T,C) -> (N,C,T)
        x = x.transpose(1, 2)
        x = self.conv1(x)
        x = self.relu(x)
        x = self.inst1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.inst2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.relu(x)
        x = self.inst3(x)
        x = self.drop3(x)

        x = self.conv4(x)
        x = self.relu(x)
        x = self.inst4(x)
        x = self.drop4(x)

        x = self.conv5(x)
        x = self.relu(x)
        x = self.inst5(x)
        x = self.drop5(x)

        x = self.conv6(x)
        x = self.relu(x)

        x = self.glap(x)
        x = x.flatten(start_dim=1)

        logits = x
        return logits

In [6]:
class ChangClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.densa = nn.Linear(in_features=100, out_features=12)

    def forward(self, x):
        logits = self.densa(x)
        return logits

In [7]:
@torch.no_grad()
def evaluate(encoder, classifier, loader, loss_fn, device):
    encoder.eval()
    classifier.eval()
    total_loss = 0
    total_samples = 0
    y_true = []
    y_pred = []

    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = classifier(encoder(X))
        loss = loss_fn(logits, y)
        total_loss += loss.item() * len(y)
        total_samples += len(y)
        pred = torch.argmax(logits, dim=1)
        y_true.append(y)
        y_pred.append(pred)

    y_true = torch.cat(y_true)
    y_pred = torch.cat(y_pred)
    f1 = multiclass_f1_score(y_pred, y_true, num_classes=12, average="macro").item()

    return total_loss / total_samples, f1

# Avaliações...

In [70]:
pasta = 'C:\\Meu Drive\\Doutorado Unicamp\\Projeto\\github\\chang-UDA-HAR\\modelos\\baseline_10_PAMAP\\'
i = 1
enc = ChangEncoder().to(device)
enc.load_state_dict(torch.load(pasta+'0_baseline_encoder_'+posis[i]+'.pth', weights_only=True))
cla = ChangClassifier().to(device)
cla.load_state_dict(torch.load(pasta+'0_baseline_classifier_'+posis[i]+'.pth', weights_only=True))

<All keys matched successfully>

In [73]:
pasta = 'C:\\Meu Drive\\Doutorado Unicamp\\Projeto\\github\\chang-UDA-HAR\\modelos\\adapt_PAMAP\\'
i, j = 1, 0
enc = ChangEncoder().to(device)
enc.load_state_dict(torch.load(pasta+posis[i]+'_adapt_encoder_'+posis[j]+'.pth', weights_only=True))
cla = ChangClassifier().to(device)
cla.load_state_dict(torch.load(pasta+posis[i]+'_adapt_classifier_'+posis[j]+'.pth', weights_only=True))

<All keys matched successfully>

In [76]:
seeds = np.array([246, 940, 653, 489, 200, 331, 464, 559, 506, 229])
inds = ydata[:,0]==j
Xt = Xdata[inds]
yt = ydata[inds][:,1]
Xt_train, Xt_test, yt_train, yt_test = train_test_split(Xt, yt, test_size=0.2, random_state=42, stratify=yt)
Xt_train = torch.tensor(Xt_train, dtype=torch.float32).to(device)
Xt_test  = torch.tensor(Xt_test, dtype=torch.float32).to(device)
yt_train = torch.tensor(yt_train, dtype=torch.long).to(device)
yt_test  = torch.tensor(yt_test, dtype=torch.long).to(device)

test_loader = DataLoader(TensorDataset(Xt_test, yt_test), batch_size=256, shuffle=False)
_, test_f1 = evaluate(enc, cla, test_loader, nn.CrossEntropyLoss(), device)
test_f1

0.33558377623558044

In [72]:
realAct = ['lying', 'sitting', 'standing', 'walking', 
           'running', 'cycling', 'Nordic', 'ascending stairs', 
           'descending stairs', 'vacuum', 'ironing', 'rope jumping']
actis_desejada = ['lying', 'sitting', 'standing', 'walking', 
           'running', 'cycling', 'Nordic', 'ascending stairs', 
           'descending stairs', 'vacuum', 'ironing', 'rope jumping']
idx = [realAct.index(act) for act in actis_desejada]
logits = cla(enc(Xt_test))
y_pred = torch.argmax(logits, dim=1).cpu().numpy()
z = confusion_matrix(yt_test.cpu().numpy(), y_pred)
z_reordenada = z[np.ix_(idx, idx)]
fig = px.imshow(
    z_reordenada, 
    x=actis_desejada, 
    y=actis_desejada, 
    text_auto=True, 
    labels=dict(x="Estimado", y="Referência", color="Quantidade"),
    width=700, height=600, 
    color_continuous_scale='Blues', 
    title="Treino {}, Teste {}, f1: {:.2f}".format(posis[i], posis[i], test_f1*100)
)
fig.update_yaxes(autorange="reversed")
fig.show()

In [77]:
realAct = ['lying', 'sitting', 'standing', 'walking', 
           'running', 'cycling', 'Nordic', 'ascending stairs', 
           'descending stairs', 'vacuum', 'ironing', 'rope jumping']
actis_desejada = ['lying', 'sitting', 'standing', 'walking', 
           'running', 'cycling', 'Nordic', 'ascending stairs', 
           'descending stairs', 'vacuum', 'ironing', 'rope jumping']
idx = [realAct.index(act) for act in actis_desejada]
logits = cla(enc(Xt_test))
y_pred = torch.argmax(logits, dim=1).cpu().numpy()
z = confusion_matrix(yt_test.cpu().numpy(), y_pred)
z_reordenada = z[np.ix_(idx, idx)]
fig = px.imshow(
    z_reordenada, 
    x=actis_desejada, 
    y=actis_desejada, 
    text_auto=True, 
    labels=dict(x="Estimado", y="Referência", color="Quantidade"),
    width=700, height=600, 
    color_continuous_scale='Blues', 
    title="Treino {}, Teste {}, f1: {:.2f}".format(posis[i], posis[j], test_f1*100)
)
fig.update_yaxes(autorange="reversed")
fig.show()

In [24]:
pasta = 'C:\\Meu Drive\\Doutorado Unicamp\\Projeto\\github\\chang-UDA-HAR\\modelos\\adapt_PAMAP\\adapt_filtros\\'
vals = [0.0001, 0.001, 0.01, 0.1, 1, 5, 10, 15, 20, 25, 30, 100]
adapt = np.zeros((3,3))
for i in range(3):
    for j in range(3):
        if i==j:
            continue
        enc2 = ChangEncoder().to(device)
        nome = posis[i]+'_adapt_encoder_'+posis[j]+'.pth'
        enc2.load_state_dict(torch.load(pasta+nome, weights_only=True))
        cla2 = ChangClassifier().to(device)
        nome = posis[i]+'_adapt_classifier_'+posis[j]+'.pth'
        cla2.load_state_dict(torch.load(pasta+nome, weights_only=True))

        inds = ydata[:,0]==j
        Xs = Xdata[inds]
        ys = ydata[inds][:,1]
        _, Xs_test, _, ys_test = train_test_split(Xs, ys, test_size=0.2, random_state=42, stratify=ys)
        Xs_test  = torch.tensor(Xs_test, dtype=torch.float32).to(device)
        ys_test  = torch.tensor(ys_test, dtype=torch.long).to(device)
        test_loader = DataLoader(TensorDataset(Xs_test, ys_test), batch_size=256, shuffle=False)
        _, test_f1 = evaluate(enc2, cla2, test_loader, nn.CrossEntropyLoss(), device)
        adapt[i,j] = test_f1

In [17]:
df = pd.DataFrame((adapt*100).round(1), columns=posis, index=posis)
df

,hand,chest,ankle
hand,0.0,41.5,28.2
chest,33.6,0.0,32.1
ankle,20.7,47.9,0.0


In [25]:
df = pd.DataFrame((adapt*100).round(1), columns=posis, index=posis)
df

,hand,chest,ankle
hand,0.0,35.9,19.2
chest,30.1,0.0,29.4
ankle,25.5,32.8,0.0
